# Assignment 14 - Feature Engineering, Encoding, Scaling & Pipelines

**Author:** PATAN KHALILULLA KHAN

**Objective:** Prepare features correctly so machine learning models can learn
effectively: feature engineering, feature encoding, feature scaling, and an
end-to-end scikit-learn pipeline. **Focus is data preparation + pipeline design,
not model optimization.**

**Libraries:** `scikit-learn`, `pandas`, `numpy` ONLY (per assignment restriction).

In [1]:
import pandas as pd
import numpy as np
import main   # <-- all task functions live in main.py (same folder)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

df_raw = pd.read_csv(main.CSV_PATH)
print("Dataset loaded:", df_raw.shape, "| Setup complete.")

Dataset loaded: (891, 12) | Setup complete.


# PART 1 - Feature Engineering


## Task 1: Creating New Features

Four new features engineered from existing columns:
- `family_size = SibSp + Parch + 1`
- `is_alone = 1 if family_size == 1 else 0`
- `fare_per_person = Fare / family_size`
- `age_group` (bucketed age: Child / Teen / YoungAdult / Adult / Senior)

In [2]:
df_eng = main.task1_create_features(df_raw)
df_eng.head()


PART 1 - TASK 1: CREATING NEW FEATURES
Existing columns identified for engineering:
  -> 'sib_sp' (siblings/spouses aboard) + 'parch' (parents/children aboard)
  -> 'fare' (ticket price) + 'age'

New features added to the DataFrame:
  family_size      : SibSp + Parch + 1
  is_alone         : 1 if family_size == 1 else 0
  fare_per_person  : Fare / family_size
  age_group        : age bucketed into Child/Teen/YoungAdult/Adult/Senior

Sample (first 8 rows, engineered columns):
   PassengerId  SibSp  Parch  family_size  is_alone     Fare  fare_per_person   Age   age_group
0            1      1      0            2         0   7.2500          3.62500  22.0  YoungAdult
1            2      1      0            2         0  71.2833         35.64165  38.0       Adult
2            3      0      0            1         1   7.9250          7.92500  26.0  YoungAdult
3            4      1      0            2         0  53.1000         26.55000  35.0  YoungAdult
4            5      0      0           

## Task 2: Handling Date & Text Features

- **Text:** `Name` -> `title` (regex), `name_length`, `name_word_count`.
- **Date:** Titanic has no date column (step skipped for Titanic, explained in
  the output); the date extraction is demonstrated on `tmdb_movies.csv`
  (`release_date` -> year / month / day / weekday).

In [3]:
df_eng = main.task2_date_text_features(df_eng)


PART 1 - TASK 2: HANDLING DATE & TEXT FEATURES
2a. TEXT FEATURES - 'Name' column (Titanic has no date column):
  title           : honorific extracted via regex from 'Name'
  name_length     : character length of 'Name'
  name_word_count : number of words in 'Name'

Sample:
                                                Name title  name_length  name_word_count
0                            Braund, Mr. Owen Harris    Mr           23                4
1  Cumings, Mrs. John Bradley (Florence Briggs Th...   Mrs           51                7
2                             Heikkinen, Miss. Laina  Miss           22                3
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)   Mrs           44                7
4                           Allen, Mr. William Henry    Mr           24                4

Title value counts:
title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Maj

# PART 2 - Feature Encoding


## Task 3: One-Hot Encoding with `pd.get_dummies()`

Applied to `Sex` and `Embarked` with `drop_first=True` (drops the alphabetically
first category - `female` and `C` become the baseline).

In [4]:
df_dummy = main.task3_get_dummies(df_eng)
df_dummy.head()


PART 2 - TASK 3: ONE-HOT ENCODING (pd.get_dummies)
Categorical columns identified: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked', 'age_group', 'title']

One-hot encoded DataFrame (drop_first=True):
   sex_male  embarked_Q  embarked_S
0      True       False        True
1     False       False       False
2     False       False        True
3     False       False        True
4      True       False        True
5      True        True       False
6      True       False        True
7      True       False        True

Shape before: (891, 2) | after: (891, 3)

Column mapping (drop_first drops the alphabetically first category):
  Sex female -> dropped (baseline) | Sex male -> sex_male
  Embarked C -> dropped | Q -> embarked_Q | S -> embarked_S


## Task 4: ColumnTransformer (Recommended Way)

`OneHotEncoder(handle_unknown='ignore', drop='first')` for categorical columns,
numerical columns passed through unchanged.

In [5]:
preprocessor, transformed = main.task4_column_transformer(df_eng)
transformed.head()


PART 2 - TASK 4: COLUMNTRANSFORMER (RECOMMENDED WAY)
Numerical features  : ['Age', 'SibSp', 'Parch', 'Fare', 'family_size', 'is_alone', 'fare_per_person']
Categorical features: ['Pclass', 'Sex', 'Embarked']

ColumnTransformer result (first 8 rows):
   Pclass_2  Pclass_3  Sex_male  Embarked_Q  Embarked_S   Age  SibSp  Parch     Fare  family_size  is_alone  fare_per_person
0       0.0       1.0       1.0         0.0         1.0  22.0    1.0    0.0   7.2500          2.0       0.0          3.62500
1       0.0       0.0       0.0         0.0         0.0  38.0    1.0    0.0  71.2833          2.0       0.0         35.64165
2       0.0       1.0       0.0         0.0         1.0  26.0    0.0    0.0   7.9250          1.0       1.0          7.92500
3       0.0       0.0       0.0         0.0         1.0  35.0    1.0    0.0  53.1000          2.0       0.0         26.55000
4       0.0       1.0       1.0         0.0         1.0  35.0    0.0    0.0   8.0500          1.0       1.0          8.05000


# PART 3 - Feature Scaling


## Task 5: Standardization (StandardScaler)

`z = (x - mean) / std` -> mean becomes ~0, std becomes ~1.

In [6]:
X_std = main.task5_standard_scaler(df_eng)
X_std.head()


PART 3 - TASK 5: STANDARDIZATION (StandardScaler)
Numerical features before scaling (describe):
          Age    Fare  fare_per_person
count  891.00  891.00           891.00
mean    29.36   32.20            19.92
std     13.02   49.69            35.84
min      0.42    0.00             0.00
25%     22.00    7.91             7.25
50%     28.00   14.45             8.30
75%     35.00   31.00            23.67
max     80.00  512.33           512.33

After StandardScaler - per-column mean and std:
                 mean       std
Age               0.0  1.000562
Fare              0.0  1.000562
SibSp             0.0  1.000562
Parch             0.0  1.000562
family_size      -0.0  1.000562
is_alone         -0.0  1.000562
fare_per_person  -0.0  1.000562

EXPLANATION:
  StandardScaler transforms each column with  z = (x - mean) / std.
  * mean  becomes ~0   -> data is centered around zero
  * std   becomes ~1   -> every feature has unit variance
  This puts all numerical features on the SAME scale

## Task 6: Normalization (MinMaxScaler)

`x' = (x - min) / (max - min)` -> every value lands in [0, 1]; comparison with
StandardScaler shown in the output.

In [7]:
X_norm = main.task6_minmax_scaler(df_eng)


PART 3 - TASK 6: NORMALIZATION (MinMaxScaler)
After MinMaxScaler - per-column min and max:
                 min  max
Age              0.0  1.0
Fare             0.0  1.0
SibSp            0.0  1.0
Parch            0.0  1.0
family_size      0.0  1.0
is_alone         0.0  1.0
fare_per_person  0.0  1.0

EXPLANATION:
  MinMaxScaler maps values with  x' = (x - min) / (max - min).
  * every value now lies in [0, 1]
  * the shape of the distribution is preserved (unlike z-score it
    is sensitive to outliers because min/max are extreme values)

COMPARISON - StandardScaler vs MinMaxScaler (Age column, first 5 rows):
   original_Age  StandardScaler  MinMaxScaler
0          22.0          -0.566         0.271
1          38.0           0.664         0.472
2          26.0          -0.258         0.321
3          35.0           0.433         0.435
4          35.0           0.433         0.435

When to use which:
  StandardScaler -> most tabular ML defaults, robust, keeps outliers visible.
  MinMaxSc

# PART 4 - Building ML Pipeline


## Task 7: Preprocessing Pipeline

Numerical pipeline (imputer + scaler) and categorical pipeline (imputer +
one-hot) combined via ColumnTransformer - fitted on RAW data with NaNs.

In [8]:
prep = main.task7_preprocessing_pipeline()


PART 4 - TASK 7: CREATE A PREPROCESSING PIPELINE
Numerical pipeline   : SimpleImputer(median) -> StandardScaler
Categorical pipeline : SimpleImputer(mode)  -> OneHotEncoder(drop='first')
Combined via ColumnTransformer.

Fitted on RAW data (with NaN). Prepared shape: (891, 12)
Sample of prepared data (first 5 rows x first 8 columns):
       0      1      2      3      4      5      6    7
0 -0.566  0.433 -0.474 -0.502  0.059 -1.232 -0.455  0.0
1  0.664  0.433 -0.474  0.787  0.059 -1.232  0.439  0.0
2 -0.258 -0.475 -0.474 -0.489 -0.561  0.812 -0.335  0.0
3  0.433  0.433 -0.474  0.421  0.059 -1.232  0.185  0.0
4  0.433 -0.475 -0.474 -0.486 -0.561  0.812 -0.331  0.0

(No NaN remains - imputation happened inside the pipeline.)


## Task 8: Full scikit-learn Pipeline (Raw Data -> Encoding -> Scaling -> Model)

Stratified train/test split, `LogisticRegression` (no hyperparameter tuning),
fit on train, predict on test - all in one pipeline object.

In [9]:
pipe, X_train, X_test, y_train, y_test, y_pred = main.task8_full_pipeline()


PART 4 - TASK 8: FULL SCIKIT-LEARN PIPELINE (RAW DATA -> MODEL)
Train/test split (stratified, random_state=42):
  X_train: (712, 10) | X_test: (179, 10)
  Train target dist:
 Survived
0    439
1    273
Name: count, dtype: int64
  Test  target dist:
 Survived
0    110
1     69
Name: count, dtype: int64

Full pipeline structure:
  Raw Data -> preprocessor (impute + scale + one-hot) -> LogisticRegression
    [preprocessor] -> ColumnTransformer
    [model] -> LogisticRegression

Test accuracy: 0.8212 (82.1%)

Classification report (test set):
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       110
           1       0.82      0.68      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.82       179


Sanity check - first 10 predictions vs true labels:
 true  pred
    0     0
    0     0
    1     0
    0     0
    1     1
  

## Task 9: Pipeline Benefits (Conceptual)

Answers are written below in markdown cells (also saved to `PIPELINE_NOTES.md`).

## PART 4 - TASK 9: Pipeline Benefits (Conceptual Answers)

### 1. Why are pipelines important in ML?

Pipelines wrap the **entire transformation + modelling workflow into one object**.
They make the process reproducible (one `fit` call runs every step in the right
order), deployable (the same object that was trained can be serialized and served),
and less error-prone (no chance of forgetting a step or applying steps in the wrong
order). They also make the code reusable: the same preprocessing blueprint can be
applied to a new dataset with a single call.

### 2. What problems do pipelines solve?

- **Data leakage** - `fit_transform` on training data but only `transform` on test
  data is handled automatically, so statistics (mean, std, most-frequent value,
  category mapping) are never learned from the test set.
- **Order-of-operations bugs** - steps always run in the declared sequence.
- **Preprocessing drift between training and serving** - the exact same fitted
  transformers are reused at prediction time.
- **Missing-value crashes** - imputation is part of the pipeline, so raw data with
  NaN can go straight in.
- **Duplicated / scattered code** - one object replaces many manual calls.
- **Unseen categories at serving time** - `handle_unknown='ignore'` keeps
  predictions working when new categories appear.

### 3. Manual preprocessing vs pipeline-based preprocessing

| Aspect | Manual preprocessing | Pipeline-based preprocessing |
|---|---|---|
| Leakage risk | High - easy to fit scalers on all data | Low - automatic fit/train, transform/test |
| Reproducibility | Depends on run order and memory | Deterministic - one object, one fit() |
| Deployment | Must re-implement every step in prod | Serialize the trained pipeline and reuse |
| Missing values | Manual fillna calls everywhere | Imputers inside the pipeline |
| New data | Must remember every step + its params | pipeline.transform(new_data) is enough |
| Code clarity | Long scripts with many steps | Short, declarative, readable |


In [10]:
main.task9_pipeline_benefits()


PART 4 - TASK 9: PIPELINE BENEFITS (CONCEPTUAL) - see PIPELINE_NOTES.md / README / notebook
## PART 4 - TASK 9: Pipeline Benefits (Conceptual Answers)

### 1. Why are pipelines important in ML?

Pipelines wrap the **entire transformation + modelling workflow into one object**.
They make the process reproducible (one `fit` call runs every step in the right
order), deployable (the same object that was trained can be serialized and served),
and less error-prone (no chance of forgetting a step or applying steps in the wrong
order). They also make the code reusable: the same preprocessing blueprint can be
applied to a new dataset with a single call.

### 2. What problems do pipelines solve?

- **Data leakage** - `fit_transform` on training data but only `transform` on test
  data is handled automatically, so statistics (mean, std, most-frequent value,
  category mapping) are never learned from the test set.
- **Order-of-operations bugs** - steps always run in the declared sequence.
- **Pre